In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
from tqdm import tqdm

save_dir = Path("/nas/cee-water/cjgleason/ted/swot-ml/data/multigraph_manual")
metadata_dir = save_dir / "metadata"

gauges = gpd.read_parquet(metadata_dir / "gauges.parquet").set_index('site_id')

subs_df = pd.read_parquet(metadata_dir / 'subbasins.parquet', columns=['site_id', 'is_gauge', 'uparea_km2', 'outlet_id'])
subs_df.index = subs_df.index.astype(str)

basin_subbasin_dict = subs_df.groupby('outlet_id').apply(lambda g: g.index.tolist(), include_groups=False).to_dict()

In [ ]:
subs_df['original_comid'] = subs_df.index

# A small % of gauges are dropped if they don't match well.
common_idx = subs_df.index.intersection(gauges.index)
subs_df.loc[common_idx, 'original_comid'] = gauges.loc[common_idx, 'COMID']

subs_df

In [ ]:
subs_df

In [ ]:
paths = Path("/nas/cee-ice/data/MERIT-BASINS/v07_to_bugfix").glob("v07_to_bugfix_pfaf1_*.csv")
bugfix_to_v07 = pd.concat([pd.read_csv(p, dtype=str) for p in paths]).set_index('COMID_bugfix')
subs_df = subs_df.merge(bugfix_to_v07, how='left', left_on='original_comid', right_index=True)
subs_df

In [ ]:
import xarray as xr

datasets = Path("/nas/cee-ice/data")
m2s_path = datasets/"MERIT-SWORD"/"ms_translate"/"mb_to_sword"
get_m2s_file = lambda pfaf2: m2s_path / f"mb_to_sword_pfaf_{pfaf2}_translate.nc"

subs_df['pfaf2'] = subs_df['original_comid'].astype(int)//int(1E6)

pfaf2_chunks = []
for pfaf2, row in tqdm(subs_df.groupby('pfaf2')):
    m2s_df = xr.open_dataset(get_m2s_file(pfaf2)).to_pandas()
    m2s_df = m2s_df.replace(0, pd.NA).dropna(how='all')
    m2s_df.index = m2s_df.index.astype(str)

    chunk = row.merge(m2s_df, how='inner', left_on='COMID_v07', right_index=True)
    chunk = chunk['sword_1'].rename('sword_id')
    pfaf2_chunks.append(chunk)
    
site2sword = pd.concat(pfaf2_chunks)
subs_df = subs_df.join(site2sword)

subs_df

In [ ]:
import pyarrow.dataset as ds
import itertools 

all_reaches = subs_df['sword_id'].dropna().to_list()

fields=[
    'reach_id', 'time','wse', 'wse_u', 'wse_r_u',
    'slope', 'slope_u', 'slope_r_u', 'slope2', 'slope2_u', 'slope2_r_u',
    'width', 'width_u',
    'area_total', 'area_tot_u', 'area_detct', 'area_det_u', 'area_wse',
    'layovr_val', 'node_dist', 'loc_offset', 'xtrk_dist',
    'reach_q', 'reach_q_b', 'dark_frac', 'ice_clim_f', 'partial_f',
    'obs_frac_n', 'xovr_cal_q', 'dry_trop_c', 'wet_trop_c', 'iono_c', 'xovr_cal_c'
]

reach_dir  = datasets / 'hydrocron' / 'reach'
list(reach_dir.glob('*.parquet'))

swot = []
for reach_file in reach_dir.glob('*_hydrocron_reach.parquet'):
    dataset = ds.dataset(reach_file, format="parquet")
    table = dataset.to_table(
        columns=fields,
        filter=(ds.field("reach_id").isin(all_reaches))
    )
    swot.append(table.to_pandas())
    
all_swot = pd.concat(swot)
all_swot = all_swot[all_swot['wse'] != -999999999999.0]
all_swot['d_wse'] = all_swot['wse'] - all_swot.groupby('reach_id')['wse'].transform('median')
all_swot['d_width'] = all_swot['width'] - all_swot.groupby('reach_id')['width'].transform('median')

all_swot = all_swot.set_index(['reach_id','time'])

In [ ]:
from data import ZarrBasinStore
store_path = Path("/scratch4/workspace/tlanghorst_umass_edu-swot-ml-zarr/zbs_batched")
store = ZarrBasinStore(store_path)

In [ ]:
def get_best_record(grp):
    return grp.sort_values('wse_u').iloc[0]
    
def get_swot_r(reach_id):
    if np.isnan(reach_id):
        return None
        
    if reach_id in all_swot.index.get_level_values('reach_id'):
        swot = all_swot.xs(reach_id, level='reach_id')
        swot.index = pd.to_datetime(swot.index).tz_localize('UTC')

        # For any duplicated dates, pick the best obs. according to uncertainty estimate.
        if any(swot.index.duplicated()):
            swot = swot.groupby(swot.index.name).apply(get_best_record)

        return swot.add_suffix('_river')


BATCH_SIZE = 10 
for basin_id, basin_group in tqdm(subs_df.groupby('outlet_id')):
    basin_subs = basin_subbasin_dict[basin_id]
    n_subs = len(basin_subs)

    for i in range(0, n_subs, BATCH_SIZE):
        batch_subs = basin_subs[i : i + BATCH_SIZE]
        
        batch_df_list = []
        for subbasin in batch_subs:
            reach_id = basin_group.loc[subbasin]['sword_id']
            swot_df = get_swot_r(reach_id)

            if swot_df is not None and not swot_df.empty:
                swot_df['subbasin'] = subbasin
                swot_df.index.name = 'date'
                batch_df_list.append(swot_df)

        # If the whole batch is empty, we technically don't need to write anything 
        # since the zarr was initialized with NaNs/empty).
        if batch_df_list:
            batch_df = pd.concat(batch_df_list)
            store.write_batch_data(basin_id, batch_df, basin_subs, batch_subs)